# SIH26158 - Single-Pass Drone Video to Georeferenced 3D Model

This notebook runs the complete SIH pipeline on a Colab GPU. Inputs are one 1080p/4K drone video and its synchronized DJI SRT or normalized GPS CSV. Outputs include PLY, LAS, GLB, OBJ, GeoTIFF, georeferencing metadata, timings, and accuracy diagnostics.

Before starting, select **Runtime > Change runtime type > T4 GPU**. Upload this project folder to `MyDrive/SIH26158/`, with input files under `MyDrive/SIH26158/input/`.

In [ ]:
# CELL 1 - verify GPU and mount Google Drive
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
assert gpu.returncode == 0, 'No GPU detected. Select a T4 GPU runtime before continuing.'
print('GPU:', gpu.stdout.strip())
PROJECT_DIR = '/content/drive/MyDrive/SIH26158'
assert os.path.isfile(f'{PROJECT_DIR}/requirements.txt'), f'Project not found at {PROJECT_DIR}'
print('Project:', PROJECT_DIR)

In [ ]:
# CELL 2 - install Conda. This intentionally restarts the runtime once.
# After the restart, continue from CELL 3; do not rerun this cell.
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
# CELL 3 - install the CUDA COLMAP build and Python dependencies
import os, shutil, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/SIH26158'
assert shutil.which('mamba'), 'Conda is missing. Run CELL 2 and wait for its runtime restart.'
pin = '/usr/local/conda-meta/pinned'
if os.path.exists(pin): os.remove(pin)
install = subprocess.run(['mamba','install','-y','-q','-c','conda-forge','colmap','libfaiss','openimageio'], text=True)
assert install.returncode == 0, 'COLMAP installation failed'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{PROJECT_DIR}/requirements.txt'], check=True)
check = subprocess.run(['colmap','-h'], capture_output=True, text=True, errors='replace')
header = (check.stdout or '') + (check.stderr or '')
print('\n'.join(header.splitlines()[:8]))
assert check.returncode == 0 and 'COLMAP' in header, 'COLMAP does not run'
assert 'CUDA' in header, 'This COLMAP build does not report CUDA support'
print('Installation complete')

In [ ]:
# CELL 4 - configure this flight
import os
PROJECT_DIR = '/content/drive/MyDrive/SIH26158'
RUN_NAME = 'demo_flight'
VIDEO = f'{PROJECT_DIR}/input/flight.mp4'       # change filename
TELEMETRY = f'{PROJECT_DIR}/input/flight.srt'   # .srt or .csv; change filename
QUALITY = 'draft'                               # draft first; full for final output
TARGET_FRAMES = 120                             # draft first; increase only after quality checks
DSM_RESOLUTION_M = 0.5
VALIDATION_DISTANCES = ''                       # optional CSV with surveyed distances
GROUND_TRUTH_TRAJECTORY = ''                    # optional time_s,x_m,y_m,z_m CSV
AI_MASK_DYNAMIC_OBJECTS = True                  # YOLO segmentation: people/vehicles/animals
RESET_WORKSPACE = True                          # clean previous temporary reconstruction
WORKSPACE = f'/content/sih26158_{RUN_NAME}'     # fast ephemeral disk
OUTPUT = f'{PROJECT_DIR}/outputs/{RUN_NAME}'    # persistent Drive output
assert os.path.isfile(VIDEO), f'Video not found: {VIDEO}'
assert os.path.isfile(TELEMETRY), f'Telemetry not found: {TELEMETRY}'
print('Video:', VIDEO)
print('Telemetry:', TELEMETRY)
print('Output:', OUTPUT)

In [ ]:
# CELL 5 - preflight, GPU monitoring, complete pipeline, and artifact verification
import os, subprocess, sys, time, shutil
command = [sys.executable, '-m', 'sih_drone_pipeline', 'run',
           '--video', VIDEO, '--telemetry', TELEMETRY,
           '--workspace', WORKSPACE, '--output', OUTPUT,
           '--quality', QUALITY, '--target-frames', str(TARGET_FRAMES),
           '--dsm-resolution', str(DSM_RESOLUTION_M)]
if AI_MASK_DYNAMIC_OBJECTS: command.append('--ai-mask-dynamic')
if VALIDATION_DISTANCES:
    command += ['--validation-distances', VALIDATION_DISTANCES]
if GROUND_TRUTH_TRAJECTORY:
    command += ['--ground-truth-trajectory', GROUND_TRUTH_TRAJECTORY]
env = os.environ.copy()
env['PYTHONPATH'] = PROJECT_DIR + os.pathsep + env.get('PYTHONPATH', '')
os.makedirs(OUTPUT, exist_ok=True)
if RESET_WORKSPACE and os.path.isdir(WORKSPACE): shutil.rmtree(WORKSPACE)
preflight = [sys.executable, '-m', 'sih_drone_pipeline', 'preflight', '--video', VIDEO, '--telemetry', TELEMETRY, '--workspace', WORKSPACE, '--output', f'{OUTPUT}/preflight.json']
assert subprocess.run(preflight, cwd=PROJECT_DIR, env=env).returncode == 0, 'Preflight failed; open preflight.json'
stop_file = '/content/sih_gpu_monitor.stop'
if os.path.exists(stop_file): os.remove(stop_file)
monitor = subprocess.Popen([sys.executable, '-m', 'sih_drone_pipeline.gpu_monitor', '--output', f'{OUTPUT}/gpu_usage.csv', '--stop-file', stop_file], cwd=PROJECT_DIR, env=env)
print(' '.join(command))
started = time.time()
try:
    result = subprocess.run(command, cwd=PROJECT_DIR, env=env)
finally:
    open(stop_file, 'w').close()
    monitor.wait(timeout=15)
    if os.path.isdir(f'{WORKSPACE}/logs'): shutil.copytree(f'{WORKSPACE}/logs', f'{OUTPUT}/logs', dirs_exist_ok=True)
    if os.path.isfile(f'{WORKSPACE}/run_report.partial.json'): shutil.copy2(f'{WORKSPACE}/run_report.partial.json', f'{OUTPUT}/run_report.partial.json')
print(f'Total notebook elapsed time: {(time.time()-started)/60:.1f} minutes')
assert result.returncode == 0, 'Pipeline failed; inspect OUTPUT/logs and run_report.partial.json'
verify = [sys.executable, '-m', 'sih_drone_pipeline', 'verify', '--output', OUTPUT]
assert subprocess.run(verify, cwd=PROJECT_DIR, env=env).returncode == 0, 'Artifact verification failed; open verification_report.json'

In [ ]:
# CELL 6 - inspect the judging metrics and output files
import json, os, glob
report = json.load(open(f'{OUTPUT}/run_report.json'))
verification = json.load(open(f'{OUTPUT}/verification_report.json'))
gpu = json.load(open(f'{OUTPUT}/gpu_usage.summary.json'))
print(json.dumps({
    'registered_images': report.get('sparse_metrics', {}).get('registered_images'),
    'mean_reprojection_error_px': report.get('sparse_metrics', {}).get('mean_reprojection_error_px'),
    'processing_minutes': round(report.get('wall_clock_seconds', 0) / 60, 2),
    'gps_validation': report.get('validation'),
    'targets': report.get('targets'),
    'artifact_checks_pass': verification.get('artifact_checks_pass'),
    'quality_checks': verification.get('quality_checks'),
    'production_ready': verification.get('production_ready'),
    'gpu_overall': gpu,
    'gpu_by_stage': verification.get('gpu_by_stage'),
}, indent=2))
print('\nOutputs:')
for path in sorted(glob.glob(f'{OUTPUT}/*')):
    print(f'  {os.path.basename(path):28s} {os.path.getsize(path)/1024/1024:9.1f} MB')

## Quality gates before presenting

- Most extracted frames should be registered.
- Mean reprojection error should normally be below 1 pixel.
- GPS alignment residual should be inspected, but it is not independent proof of accuracy.
- Add surveyed distances using `examples/validation_distances.example.csv`; only those checks can prove surface accuracy.
- The draft profile must be timed on the final 10-minute test video.
- Download `model.glb` and `viewer_metadata.json`, then open both in the metric web viewer.